# SHA-256 Implementation

## Problem 0: GitHub Issues

## Problem 1: Representing SHA-256 Data

### 32-bit words

FIPS 180-4 section 2.1 defines a **word** as "a group of either 32 bits or 64 bits, depending on the algorithm. For SHA-256 specifically, section 3.2 states the algorithm operates on 32-bit words.

Source: [FIPS 180-4 - Secure Hash Standard](https://csrc.nist.gov/pubs/fips/180-4/upd1/final)

In [1]:
import numpy as np

# The largest value a 32-bit word can hold, per FIPS 180-4 Sec 3.2
max_32_bit = 2**32 -1
print("Max 32-bit value:", max_32_bit)

Max 32-bit value: 4294967295


### Sequences of 32-bit words 

A few parts of SHA-256 need more than just one word - a message block consists of sixteen words. For this we must find a way to put words together in a sequence.

From my research, a NumPy array seemed the best fit over a plain list. Every value in the array is forced to be a 32-bit word, which matches how the standard treats every value. It is also easy to apply an operation to the whole sequence in this format.

- [How to create an array and specify its dtype](https://numpy.org/doc/stable/reference/generated/numpy.array.html)

- [NumPy wraparound/overflow behaviour](https://numpy.org/doc/stable/user/basics.types.html)


In [2]:
# a sequence of 32-bit words
words = np.array([1,2,3,4], dtype=np.uint32)
print(words)
print(words.dtype)


[1 2 3 4]
uint32


In [3]:
# checking the whole array still wraps around like a 32-bit word
print(words +np.uint32(2**32 -1))

[0 1 2 3]


### Input messages

FIPS 180-4 works on bits and bytes, not text. In python, a string represents Unicode text, not raw bytes, so it can't be hashed directly, it needs to be converted first.

I will represent the input message as a Python bytes object. If the message starts out as a string, I can convert it using .encode(), which turns it into its raw byte representation (UTF-8 by default).

- [Python docs - Text vs Data](https://docs.python.org/3/library/stdtypes.html#text-sequence-type-str)

In [4]:
# a string is text, not raw bytes - can't be hashed directly 
message = "abc"
print(type(message))

<class 'str'>


In [5]:
# converting to bytes using UTF-8 encoding
message_bytes = message.encode("utf-8")
print(message_bytes)
print(type(message_bytes))

b'abc'
<class 'bytes'>


In [6]:
# non-ASCII chracters can take more than one byte once encoded
message2 = "café"
print(len(message2))  #character count
print(len(message2.encode("utf-8")))   #byte count - different

4
5


### 512-bit message blocks

FIPS 180-4 section 3.1(4a) defines a SHA-256 message block as 512 bits, 
represented as a sequence of sixteen 32-bit words.

This representation is appropriate because a 64-byte slice of the message 
(512 bits / 8 = 64 bytes) gives the correct block size, and splitting that 
slice into sixteen 4-byte groups gives exactly the word count the standard 
specifies - reusing the same numpy.uint32 array representation I already 
justified for a sequence of words.

**References**
- [FIPS 180-4 - Secure Hash Standard](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) - section 3.1(4a)
- [Python docs - Common Sequence Operations](https://docs.python.org/3/library/stdtypes.html#common-sequence-operations) - slicing syntax
- [Python docs - int.from_bytes()](https://docs.python.org/3/library/stdtypes.html#int.from_bytes) - byte-to-integer conversion

In [7]:
# 64 bytes of distinct values (0-63), so each word is easy to verify by eye
message = bytes(range(64))
print(message)
print(len(message), "bytes - exactly one 512-bit block")

b'\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\x0c\r\x0e\x0f\x10\x11\x12\x13\x14\x15\x16\x17\x18\x19\x1a\x1b\x1c\x1d\x1e\x1f !"#$%&\'()*+,-./0123456789:;<=>?'
64 bytes - exactly one 512-bit block


In [8]:
# slicing out the first 64-byte block
block = message[0:64]
print(len(block), "bytes")

64 bytes


In [9]:
# split the 64-byte block into sixteen 4-byte groups, one word each
words = np.array(
    [int.from_bytes(block[i:i + 4], byteorder="big") for i in range(0, 64, 4)],
    dtype=np.uint32,
)
print(words)
print(len(words), "words")
print(words.dtype)

[     66051   67438087  134810123  202182159  269554195  336926231
  404298267  471670303  539042339  606414375  673786411  741158447
  808530483  875902519  943274555 1010646591]
16 words
uint32


The first word above is 66051, which matches a manual check: bytes 0-3 
are 0x00, 0x01, 0x02, 0x03, and big-endian reads that as 0x00010203 = 66051. 
This confirms the representation correctly preserves both the byte order 
FIPS 180-4 specifies and the word/array structure needed for the later 
compression function.

### Big-endian integers

[FIPS 180-4 section 3.1(2)](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
says the standard uses "the 'big-endian' convention... so that within each 
word, the most significant bit is stored in the left-most bit position."

Basically, endianness is about which end of a group of bytes you treat as 
the "start" when reading it as one number - and the same bytes can give 
two completely different values depending on which way you read them. 
Python's `int.from_bytes()` has a `byteorder` argument for exactly this, 
which explains why I've been writing `byteorder="big"` in my earlier cells 



In [10]:
# the same four bytes, interpreted two different ways
sample_bytes = bytes([0x00, 0x01, 0x02, 0x03])

big_endian_value = int.from_bytes(sample_bytes, byteorder="big")
little_endian_value = int.from_bytes(sample_bytes, byteorder="little")

print("Big-endian:", big_endian_value)
print("Little-endian:", little_endian_value)

Big-endian: 66051
Little-endian: 50462976


In [11]:
# FIPS 180-4 section 3.1(3) gives 291 as an example, represented in hex as "00000123"
fips_example_bytes = bytes.fromhex("00000123")
result = int.from_bytes(fips_example_bytes, byteorder="big")
print(result)  # should be 291, matching the standard's own example

291


## Problem 2: SHA-256 Bitwise Operations

### Ch(x, y, z)

[FIPS 180-4 equation 4.2](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this function as:

Ch(x, y, z) = (x AND y) XOR (NOT x AND z)

x controls which of the other two values gets 
picked, one bit at a time. If a bit in x is 1, the matching bit from y is 
used. If a bit in x is 0, the matching bit from z is used instead. This happens separately for all 32 bits at once.

I implemented this directly using Python's bitwise operators - `&` for 
AND, `^` for XOR, and `~` for NOT. These are 
[the same operators NumPy uses](https://numpy.org/doc/stable/reference/routines.bitwise.html), 
and they work on each bit of the number individually, which is exactly what this function needs.

In [4]:
def ch(x, y, z):
    """
    SHA-256 Ch (choose) function.

    For each bit position, selects the bit from y where x is 1,
    and the bit from z where x is 0.

    Parameters
    ----------
    x, y, z : numpy.uint32
        32-bit unsigned integer inputs.

    Returns
    -------
    numpy.uint32
        Result of (x AND y) XOR (NOT x AND z).
    """
    return (x & y) ^ (~x & z)

In [6]:
x_all_ones = np.uint(0xFFFFFFFF)
z_all_zeros = np.uint(0x00000000)
y_test = np.uint(0x12345678)
z_test = np.uint(0x87654321)

assert ch(x_all_ones, y_test, z_test) == y_test
assert ch(z_all_zeros, y_test, z_test) == z_test
print("Identity tests passed")

Identity tests passed


In [7]:
x_mixed = np.uint32(0x0000000F)
y_mixed = np.uint32(0x000000F0)
z_mixed = np.uint32(0x000000AA)

result = ch(x_mixed, y_mixed, z_mixed)
print(hex(result))
assert result == np.uint32(0xA0)

0xa0


### Maj(x, y, z)

[FIPS 180-4 equation 4.3](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this function as:

Maj(x, y, z) = (x AND y) XOR (x AND z) XOR (y AND z)

At each bit position, this looks at the three 
matching bits from x, y and z, and picks whichever value shows up most - 
a majority vote done separately for every single bit. If two or more of 
the three bits are 1, the output bit is 1. Otherwise it's 0.

I implemented this directly using `&` for AND and `^` for XOR, the same 
[bitwise operators](https://numpy.org/doc/stable/reference/routines.bitwise.html) 
I used for Ch.

In [8]:
def maj(x, y, z):
    """
    SHA-256 Maj (majority) function.

    For each bit position, returns the majority value among
    x, y, and z (the bit that appears in at least two of the three).

    Parameters
    ----------
    x, y, z : numpy.uint32
        32-bit unsigned integer inputs.

    Returns
    -------
    numpy.uint32
        Result of (x AND y) XOR (x AND z) XOR (y AND z).
    """
    return (x & y) ^ (x & z) ^ (y & z)

In [9]:
same_value = np.uint32(0x5A5A5A5A)
assert maj(same_value, same_value, same_value) == same_value
print("All-equal test passed")

All-equal test passed


In [10]:
x_mixed = np.uint32(0x0000000F)
y_mixed = np.uint32(0x000000F0)
z_mixed = np.uint32(0x000000AA)

result = maj(x_mixed, y_mixed, z_mixed)
print(hex(result))
assert result == np.uint32(0xAA)

0xaa


### ROTR (rotate right)

[FIPS 180-4 section 3.2](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this operation as:

ROTR^n(x) = (x >> n) OR (x << (32 - n))

Every bit in x moves n places to the right. 
Any bit that falls off the right end wraps around and comes back in at 
the left end, so no bits are lost. This is also called a 
[circular shift](https://en.wikipedia.org/wiki/Circular_shift). Unlike a 
normal shift, the empty spaces are filled with the bits that fell off 
instead of zeros. 
[Baeldung's guide to bit shift operators](https://www.baeldung.com/cs/bitwise-shift-operators) 
explains the same difference, and a 
[paper on bit rotation](https://jesit.springeropen.com/articles/10.1186/s43067-021-00029-8) 
mentions that rotations are used in cryptography, which fits with SHA-256 
using them.

I implemented this directly using the 
[right shift and left shift operators](https://numpy.org/doc/stable/reference/generated/numpy.right_shift.html) 
(`>>` and `<<`) and `|` for OR. Because the values are uint32, bits shifted 
off the left end are dropped, which is what the formula relies on.

In [2]:
def rotr(x, n):
    """
    Rotate a 32-bit word right by n positions (FIPS 180-4 section 3.2).

    Parameters
    ----------
    x : numpy.uint32
        The 32-bit word to rotate.
    n : int
        Number of positions to rotate, 0 < n < 32.

    Returns
    -------
    numpy.uint32
        x rotated right by n bits.
    """
    n = np.uint32(n)
    return (x >> n) | (x << (np.uint32(32) - n))

In [3]:
assert rotr(np.uint32(0x00000001), 1) == np.uint32(0x80000000)
assert rotr(np.uint32(0x12345678), 8) == np.uint32(0x78123456)
print("rotr tests passed")

rotr tests passed


### SHR (shift right)

[FIPS 180-4 section 3.2](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this operation as:

SHR^n(x) = x >> n

Every bit in x moves n places to the right, 
the same as ROTR. The difference is that bits falling off the right end 
are thrown away, and zeros fill in on the left. This is known as a 
[logical shift](https://en.wikipedia.org/wiki/Logical_shift), and 
[Baeldung](https://www.baeldung.com/cs/bitwise-shift-operators) describes 
it the same way, padding the empty places with zeros.

I implemented this directly using the `>>` operator, which does exactly 
this on a uint32.

In [5]:
def shr(x, n):
    """
    Shift a 32-bit word right by n positions (FIPS 180-4 section 3.2).

    Parameters
    ----------
    x : numpy.uint32
        The 32-bit word to shift.
    n : int
        Number of positions to shift, 0 <= n < 32.

    Returns
    -------
    numpy.uint32
        x shifted right by n bits, with zeros filled in on the left.
    """
    return x >> np.uint32(n)

In [6]:
assert shr(np.uint32(0x00000001), 1) == np.uint32(0x00000000)
assert shr(np.uint32(0x0000000F), 1) == np.uint32(0x00000007)
print("shr tests passed")

shr tests passed


## Problem 3: Generating the SHA-256 Constants

## Problem 4: Padding and Parsing Messages

## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256